# ExRec Logical Error Distribution
## Syndrome-conditioned logical error analysis for full and truncated exRecs

**Circuit structures analysed:**
- **Full exRec**: `s_in ──► FTEC_a ──► [fault_0] ──► GATE ──► [fault_1] ──► FTEC_b ──► (L, s_out)`
- **Truncated exRec**: `s_in ──► FTEC_a ──► [fault_0] ──► GATE ──► *(star) ──► (L, s_out)`

The star decoder `*` measures the syndrome of whatever error is present and applies a virtual correction — its output syndrome `s_out` feeds into the next exRec as `s_in`.

**Key question**: for a sampled fault path, how does the input syndrome `s_in` determine the probability of each logical error `{I, X, Y, Z}`?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from pauli_utils import (
    LOGICAL_NAMES,
    pauli_str, str_to_pauli, pauli_weight, all_n_qubit_paulis,
    gate_identity, gate_hadamard, gate_phase_S, gate_phase_Sdg, make_custom_gate,
)
from StabilizerCode import StabilizerCode


## User Input
Configure the code, gate, and simulation parameters here.

| Variable | Description |
|---|---|
| `code` | StabilizerCode instance to simulate |
| `gate_action` | Symplectic gate callable (identity, hadamard, phase_S, or custom) |
| `p_val` | Single-qubit depolarizing error rate |
| `N_SAMPLES` | Monte Carlo sample count (increase for better statistics) |


In [ ]:
# ---------------------------------------------------------------------------
# [[5,1,3]] perfect code  (Gottesman Table 3.2 / 3.4)
# ---------------------------------------------------------------------------
# Generators:  g1=XZZXI  g2=IXZZX  g3=XIXZZ  g4=ZXIXZ
# Logical X̄ = XXXXX,  Logical Z̄ = ZZZZZ

code_513 = StabilizerCode(
    n=5,
    generators=[
        (np.array([1,0,0,1,0]), np.array([0,1,1,0,0])),  # g1: XZZXI
        (np.array([0,1,0,0,1]), np.array([0,0,1,1,0])),  # g2: IXZZX
        (np.array([1,0,1,0,0]), np.array([0,0,0,1,1])),  # g3: XIXZZ
        (np.array([0,1,0,1,0]), np.array([1,0,0,0,1])),  # g4: ZXIXZ
    ],
    logical_x=(np.array([1,1,1,1,1]), np.array([0,0,0,0,0])),  # XXXXX
    logical_z=(np.array([0,0,0,0,0]), np.array([1,1,1,1,1])),  # ZZZZZ
    name='[[5,1,3]]',
)


# ---------------------------------------------------------------------------
# *** CHANGE INPUTS HERE ***
# ---------------------------------------------------------------------------
code      = code_513        # choose: code_513 | code_311 | your own StabilizerCode
gate      = gate_identity   # choose: gate_identity | gate_hadamard | gate_phase_S | make_custom_gate(...)
p_val     = 1e-3            # single-qubit depolarizing rate
N_SAMPLES = 500_000         # MC samples (increase for smoother distributions)
# ---------------------------------------------------------------------------

print(code)
print(f"  Syndrome table: {len(code.corrections)} entries")


## Core function: `get_logical_error`

The central function of this notebook.  Given:
- a code,
- an input syndrome `s_in`,
- a fault path of **three** data-error Paulis (Error1, Error2, Error3),
- the intended logical gate,

it returns `(L, s_out)` where `L ∈ {I,X,Y,Z}` is the logical error and `s_out` is the output syndrome.

**Circuit structure:**

```
Q_{s_in} · Error1 ──► EC ──► Error2 ──► GATE ──► Error3 ──► EC ──► (L, s_out)
```

**Fault path convention:**

| Index | Fault location |
|---|---|
| `fault_path[0]` | Error1 — 5-qubit Pauli error multiplied onto the `Q_{s_in}` residual |
| `fault_path[1]` | Error2 — error between first EC output and gate input |
| `fault_path[2]` | Error3 — error between gate output and second EC input |

`Q_{s_in}` is the minimum-weight correction for syndrome `s_in`.  The first EC
measures the syndrome of `Q_{s_in} · Error1` and applies a minimum-weight
correction.  The second EC does the same on the accumulated error after the gate.

In [ ]:
def get_logical_error(code, s_in, fault_path, gate_action):
    """
    Compute the logical error and output syndrome for one exRec.

    Circuit:  Q_{s_in} . Error1 -> EC -> Error2 -> GATE -> Error3 -> EC

    Parameters
    ----------
    code : StabilizerCode
    s_in : int
        Incoming syndrome.  The minimum-weight correction Q_{s_in} is the
        residual Pauli carried forward from previous circuitry.
    fault_path : list of (xvec, zvec), length 3
        fault_path[0] : Error1 -- data error multiplied onto Q_{s_in} before first EC.
        fault_path[1] : Error2 -- data error between first EC and gate.
        fault_path[2] : Error3 -- data error between gate and second EC.
    gate_action : callable  (xvec, zvec) -> (xvec', zvec')
        Symplectic action of the intended gate on a data-error Pauli.

    Returns
    -------
    L     : int   0=I, 1=X, 2=Y, 3=Z
    s_out : int   output syndrome (in [0, code.n_syndromes) )
    """
    # --- Step 1: Start with Q_{s_in} (min-weight correction for incoming syndrome) ---
    Qx, Qz = code.corrections[s_in]
    xE, zE = Qx.copy(), Qz.copy()

    # --- Step 2: Multiply Error1 onto Q_{s_in} ---
    xf0, zf0 = fault_path[0]
    xE, zE = xE ^ xf0, zE ^ zf0

    # --- Step 3: First EC ---
    # Measure syndrome of Q_{s_in} . Error1 and apply min-weight correction.
    s_mid = code.compute_syndrome(xE, zE)
    Cx, Cz = code.corrections[s_mid]
    xE, zE = xE ^ Cx, zE ^ Cz

    # --- Step 4: Error2 accumulates on the data ---
    xf1, zf1 = fault_path[1]
    xE, zE = xE ^ xf1, zE ^ zf1

    # --- Step 5: Gate conjugates the accumulated error ---
    xE, zE = gate_action(xE, zE)

    # --- Step 6: Error3 accumulates on the data ---
    xf2, zf2 = fault_path[2]
    xE, zE = xE ^ xf2, zE ^ zf2

    # --- Step 7: Second EC ---
    # Measure syndrome and apply min-weight correction.
    s_out = code.compute_syndrome(xE, zE)
    Rx, Rz = code.corrections[s_out]
    xRes, zRes = xE ^ Rx, zE ^ Rz

    # --- Step 8: Logical class of the zero-syndrome residual ---
    L, _ = code.logical_class(xRes, zRes)

    return L, s_out


# ---------------------------------------------------------------------------
# Worked example
# ---------------------------------------------------------------------------
_s_in = 1
_Error1  = str_to_pauli('IXIII')   # Error1
_Error2  = str_to_pauli('IIIII')   # Error2
_Error3  = str_to_pauli('IIIII')   # Error3

L_ex, s_out_ex = get_logical_error(
    code_513, _s_in,
    fault_path=[_Error1, _Error2, _Error3],
    gate_action=gate_identity,
)
print(f"Worked example:  s_in={_s_in}, Error1=IXIII, Error2=IIIII, Error3=IIIII, gate=identity")
print(f"  -> logical error = {LOGICAL_NAMES[L_ex]},  s_out = {s_out_ex}")

## Given a Fault path, distribution of logical error depending on s_in

In [ ]:
def logical_error_dist_per_fault_path(code, fault_path, gate_action):
    error_list = []
    p_id = 0
    p_x = 0
    p_y = 0
    p_z = 0
    num_syn = len(code.corrections)
    for i in range(num_syn):
        L_error, s_out = get_logical_error(code, i, fault_path, gate_action)
        error_list.append(L_error)
    fig, ax = plt.subplots(1,2,figsize = (12,5))
    syn = list(range(num_syn))
    ax[0].plot(syn, error_list, "o")
    ax[0].plot(syn, [0 for _ in range(num_syn)], label = "I")
    ax[0].plot(syn, [1 for _ in range(num_syn)], label = "X")
    ax[0].plot(syn, [2 for _ in range(num_syn)], label = "Y")
    ax[0].plot(syn, [3 for _ in range(num_syn)], label = "Z")
    ax[0].legend()
    ax[0].set_xlabel("Syndrome")
    ax[0].set_ylabel("Pauli")
    ax[0].set_title("Error for each syndrome given fault path")

    ax[1].hist(error_list)
    ax[1].set_xlabel("Pauli")
    ax[1].set_ylabel("Occurence")
    ax[1].set_title("Histogram of Logical Pauli Errors")
    plt.show()




In [ ]:
error1 = str_to_pauli("IXIII")
error2 = str_to_pauli("IIIII")
error3 = str_to_pauli("IIXII")

fault_path = [error1, error2, error3]

logical_error_dist_per_fault_path(code, fault_path, gate_identity)

In [ ]:
def sample_fault_path(code, p, n_locations, rng=None):
    """
    Sample one fault path consisting of `n_locations` independent n-qubit Pauli errors.

    Parameters
    ----------
    code        : StabilizerCode
    p           : float  — single-qubit depolarizing rate
    n_locations : int    — 1 for truncated exRec, 2 for full exRec
    rng         : np.random.Generator, optional

    Returns
    -------
    fault_path : list of (xvec, zvec), length n_locations
    """
    if rng is None:
        rng = np.random.default_rng()
    n = code.n
    fault_path = []
    for _ in range(n_locations):
        xv = np.zeros(n, dtype=int)
        zv = np.zeros(n, dtype=int)
        for q in range(n):
            r = rng.random()
            if r < p / 3:
                xv[q], zv[q] = 1, 0   # X
            elif r < 2 * p / 3:
                xv[q], zv[q] = 1, 1   # Y
            elif r < p:
                xv[q], zv[q] = 0, 1   # Z
            # else: I (do nothing)
        fault_path.append((xv, zv))
    return fault_path


def monte_carlo_distribution(code, gate_action, p, s_in,
                              n_samples=500_000, seed=None):
    """
    Estimate P(L, s_out | s_in) by Monte Carlo for a *fixed* input syndrome.

    The error model uses 3 fault locations:
        Q_{s_in} . Error1 -> EC -> Error2 -> GATE -> Error3 -> EC

    For each sample:
      1. Sample a random fault path (3 locations) under depolarizing(p).
      2. Call get_logical_error with the given s_in and record (s_out, L).

    Parameters
    ----------
    code        : StabilizerCode
    gate_action : callable
    p           : float
    s_in        : int       -- fixed input syndrome
    n_samples   : int
    seed        : int or None

    Returns
    -------
    counts_L    : np.ndarray shape (4,)  -- raw counts of each logical error
    counts_sout : np.ndarray shape (n_syn,)  -- raw counts of each output syndrome
    """
    rng   = np.random.default_rng(seed)
    n_syn = code.n_syndromes
    n_loc = 3

    counts_L    = np.zeros(4, dtype=np.int64)
    counts_sout = np.zeros(n_syn, dtype=np.int64)

    for _ in range(n_samples):
        fp = sample_fault_path(code, p, n_loc, rng=rng)
        L, s_out = get_logical_error(code, s_in, fp, gate_action)
        counts_L[L] += 1
        counts_sout[s_out] += 1

    return counts_L, counts_sout


In [ ]:
#Example

s_in = 5
p = 1e-1



count_L , count_sout = monte_carlo_distribution(code_513, gate_identity, p, s_in)

print(count_L/count_L.sum())
print(count_sout/count_sout.sum())

In [ ]:
# --- Run MC for all 16 syndromes and plot stacked P(X), P(Y), P(Z) ---

n_syn = code.n_syndromes
probs = np.zeros((n_syn, 4))  # columns: I, X, Y, Z

for s in range(n_syn):
    counts_L, _ = monte_carlo_distribution(
        code, gate, p=p_val, s_in=s, n_samples=N_SAMPLES, seed=42 + s
    )
    probs[s] = counts_L / counts_L.sum()
    print(f's_in={s:2d}  P(I)={probs[s,0]:.6f}  P(X)={probs[s,1]:.6f}  '
          f'P(Y)={probs[s,2]:.6f}  P(Z)={probs[s,3]:.6f}')

# --- Stacked bar chart ---
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(n_syn)
px, py, pz = probs[:, 1], probs[:, 2], probs[:, 3]

ax.bar(x, px, label='P(X)', color='#e74c3c')
ax.bar(x, py, bottom=px, label='P(Y)', color='#2ecc71')
ax.bar(x, pz, bottom=px + py, label='P(Z)', color='#3498db')

ax.set_xlabel('Input syndrome $s_{in}$')
ax.set_ylabel('Logical error probability')
ax.set_title(f'Logical error distribution per syndrome  '
             f'({code.name}, {gate.__name__}, p={p_val})')
ax.set_xticks(x)
ax.legend()
plt.tight_layout()
plt.show()

## Fault path sampling
Under independent depolarizing noise with single-qubit rate `p`, each qubit at each fault location has an `I` error with probability `1-p` and each of `X, Y, Z` with probability `p/3`.

In [ ]:
_NON_ID = [(1,0), (1,1), (0,1)]   # X, Y, Z as (x_bit, z_bit)


def depolarizing_prob(xv, zv, p):
    """Probability of Pauli (xv,zv) under independent single-qubit depolarizing(p)."""
    w = int(np.sum(xv | zv))
    n = len(xv)
    return (p / 3) ** w * (1 - p) ** (n - w)





def enumerate_fault_paths(code, n_locations, max_weight=1):
    """
    Enumerate all fault paths up to total Pauli weight `max_weight`.

    Each fault path is yielded as (fault_path, weight) where
    fault_path is a list of (xvec, zvec) of length n_locations.

    Useful for exact low-order probability calculations.
    """
    n = code.n
    zero_n = (np.zeros(n, dtype=int), np.zeros(n, dtype=int))
    # All 4^n Paulis for a single location
    all_paulis = all_n_qubit_paulis(n)

    # Build all combinations of n_locations Paulis with total weight ≤ max_weight
    # (identity = weight 0, each non-identity Pauli contributes its weight)
    def _rec(loc, remaining_weight, current):
        if loc == n_locations:
            yield list(current)
            return
        for xv, zv in all_paulis:
            w = int(np.sum(xv | zv))
            if w <= remaining_weight:
                current.append((xv.copy(), zv.copy()))
                yield from _rec(loc + 1, remaining_weight - w, current)
                current.pop()

    for fp in _rec(0, max_weight, []):
        total_w = sum(int(np.sum(xv | zv)) for xv, zv in fp)
        yield fp, total_w

## Exact distribution via enumeration
For each fault path weighted by its depolarizing probability, accumulate `P(L, s_out | s_in)` exactly.
This gives the full syndrome-conditioned logical error distribution up to a chosen Pauli weight.

In [ ]:
def compute_logical_error_distribution(code, gate_action, p, max_weight=None):
    """
    Compute P(L, s_out | s_in) exactly by iterating over ALL fault paths.
    WARNING: exponential in n*n_locations -- only feasible for small codes or low max_weight.

    The error model uses 3 fault locations:
        Q_{s_in} . Error1 -> EC -> Error2 -> GATE -> Error3 -> EC

    Returns
    -------
    dist : np.ndarray, shape (n_syndromes, n_syndromes, 4)
        dist[s_in, s_out, L] = P(L, s_out | s_in)
    """
    n_loc  = 3
    n_syn  = code.n_syndromes
    n      = code.n

    if max_weight is None:
        all_paulis = all_n_qubit_paulis(n)
        from itertools import product as iproduct
        fault_paths = list(iproduct(all_paulis, repeat=n_loc))
    else:
        fault_paths = [fp for fp, _ in enumerate_fault_paths(code, n_loc, max_weight=max_weight)]

    dist = np.zeros((n_syn, n_syn, 4))
    for fp in fault_paths:
        fp_list = list(fp)
        prob = 1.0
        for xv, zv in fp_list:
            prob *= depolarizing_prob(xv, zv, p)
        if prob == 0.0:
            continue
        for s_in in range(n_syn):
            L, s_out = get_logical_error(code, s_in, fp_list, gate_action)
            dist[s_in, s_out, L] += prob

    row_sums = dist.sum(axis=(1, 2), keepdims=True)
    dist /= np.where(row_sums == 0, 1, row_sums)
    return dist


def marginal_L_given_sin(dist):
    """
    Marginalise P(L, s_out | s_in) over s_out to get P(L | s_in).

    Parameters
    ----------
    dist : np.ndarray  shape (n_syn, n_syn, 4)

    Returns
    -------
    marginal : np.ndarray  shape (n_syn, 4)
    """
    return dist.sum(axis=1)

## Monte Carlo sampling
For large codes or higher fault-path weights, exact enumeration becomes expensive.  The Monte Carlo version samples fault paths from the depolarizing distribution and estimates `P(L | s_in)` by counting.

## Visualization: correlation between `s_in` and logical error type

### Plot 1 — `P(L | s_in)` heatmap (full vs truncated exRec)
### Plot 2 — `P(error | s_in)` bar chart, coloured by error type
### Plot 3 — `s_out` distribution as a function of `s_in` (syndrome propagation)

In [ ]:
def plot_logical_error_heatmaps(marg_full, marg_trunc, code, p, gate_name='identity'):
    """
    Side-by-side heatmaps of P(L | s_in) for full and truncated exRec.
    """
    n_syn = code.n_syndromes
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    titles = ['Full exRec  (FTEC–GATE–FTEC)', 'Truncated exRec  (FTEC–GATE)']
    for ax, title, marg in zip(axes, titles, [marg_full, marg_trunc]):
        im = ax.imshow(marg, aspect='auto', cmap='viridis', vmin=0, vmax=marg.max() * 1.05)
        ax.set_xticks(range(4))
        ax.set_xticklabels(['I', 'X', 'Y', 'Z'], fontsize=12)
        ax.set_yticks(range(n_syn))
        ax.set_yticklabels([f'{s:0{code.r}b}' for s in range(n_syn)], fontsize=8)
        ax.set_xlabel('Logical error  L', fontsize=11)
        ax.set_ylabel('Input syndrome  s_in', fontsize=11)
        ax.set_title(f'{title}\np = {p},  gate = {gate_name}', fontsize=11)
        plt.colorbar(im, ax=ax, fraction=0.04)
    plt.tight_layout()
    plt.show()


def plot_error_rate_by_syndrome(marg_full, marg_trunc, code, p, gate_name='identity'):
    """
    Stacked bar chart: P(X|s_in), P(Y|s_in), P(Z|s_in) per syndrome.
    """
    n_syn  = code.n_syndromes
    x      = np.arange(n_syn)
    colors = {'X': '#e74c3c', 'Y': '#9b59b6', 'Z': '#3498db'}
    fig, axes = plt.subplots(1, 2, figsize=(16, 4), sharey=True)
    for ax, title, marg in zip(axes,
                                ['Full exRec', 'Truncated exRec'],
                                [marg_full, marg_trunc]):
        pX, pY, pZ = marg[:, 1], marg[:, 2], marg[:, 3]
        ax.bar(x, pX, label='P(X)', color=colors['X'], alpha=0.85)
        ax.bar(x, pY, bottom=pX, label='P(Y)', color=colors['Y'], alpha=0.85)
        ax.bar(x, pZ, bottom=pX+pY, label='P(Z)', color=colors['Z'], alpha=0.85)
        ax.set_xticks(x)
        ax.set_xticklabels([f'{s:0{code.r}b}' for s in range(n_syn)],
                           rotation=45, ha='right', fontsize=8)
        ax.set_xlabel('Input syndrome  s_in  (binary)', fontsize=10)
        ax.set_ylabel('P(logical error)', fontsize=10)
        ax.set_title(f'{title}  [p={p}, gate={gate_name}]', fontsize=10)
        ax.legend(fontsize=9)
        ax.yaxis.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_syndrome_propagation(dist_full, dist_trunc, code, p, gate_name='identity'):
    """
    Heatmap of P(s_out | s_in) marginalised over L.
    """
    n_syn = code.n_syndromes
    prop_full  = dist_full.sum(axis=2)   # (n_syn, n_syn)
    prop_trunc = dist_trunc.sum(axis=2)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, title, prop in zip(axes, ['Full exRec', 'Truncated exRec'],
                                [prop_full, prop_trunc]):
        im = ax.imshow(prop, aspect='auto', cmap='Blues', vmin=0, vmax=1)
        labels = [f'{s:0{code.r}b}' for s in range(n_syn)]
        ax.set_xticks(range(n_syn)); ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=7)
        ax.set_yticks(range(n_syn)); ax.set_yticklabels(labels, fontsize=7)
        ax.set_xlabel('Output syndrome  s_out', fontsize=10)
        ax.set_ylabel('Input syndrome  s_in', fontsize=10)
        ax.set_title(f'P(s_out | s_in) — {title}\np={p}, gate={gate_name}', fontsize=10)
        plt.colorbar(im, ax=ax, fraction=0.04)
    plt.tight_layout()
    plt.show()


# -----------------------------------------------------------------------
# Compute truncated exRec via MC and plot
# -----------------------------------------------------------------------
print(f'Running Monte Carlo  (truncated exRec, identity gate, {N_SAMPLES} samples)...')
dist_trunc, _ = monte_carlo_distribution(
    code, gate, p=p_val,
    exrec_type='truncated', n_samples=N_SAMPLES, seed=43
)
marg_trunc = marginal_L_given_sin(dist_trunc)

plot_logical_error_heatmaps(marg_full, marg_trunc, code, p=p_val)
plot_error_rate_by_syndrome(marg_full, marg_trunc, code, p=p_val)
plot_syndrome_propagation(dist_full, dist_trunc, code, p=p_val)


## Non-Markovianity measure
A quantitative measure of how much the logical error probabilities depend on `s_in`. If the exRec were Markovian (abstract EC limit), `P(L | s_in)` would be independent of `s_in`.

In [ ]:
def non_markovianity(marg):
    """
    Scalar non-Markovianity measure: average total-variation distance of
    each P(L | s_in) row from the syndrome-averaged marginal.

       NM = (1/n_syn) * Σ_{s_in}  (1/2) * Σ_L |P(L|s_in) - P_avg(L)|

    Returns NM → 0 in the Markovian (abstract EC) limit.
    """
    p_avg = marg.mean(axis=0)
    tvd   = 0.5 * np.abs(marg - p_avg[None, :]).sum(axis=1)
    return tvd.mean(), p_avg


# -----------------------------------------------------------------------
# NM table for the four gate/exrec combinations already computed above
# -----------------------------------------------------------------------
print('Non-Markovianity  NM  (avg TVD from syndrome-average)')
print(f'{"Config":<40}  {"NM":>10}  {"P_avg(I)":>10}  {"P_avg(X)":>8}  {"P_avg(Y)":>8}  {"P_avg(Z)":>8}')
print('-' * 90)
for (gname, rtype), res in results.items():
    nm, p_avg = non_markovianity(res['marg'])
    print(f'  gate={gname:<10}  exrec={rtype:<10}   '
          f'{nm:10.2e}   {p_avg[0]:10.6f}  {p_avg[1]:8.6f}  {p_avg[2]:8.6f}  {p_avg[3]:8.6f}')

# -----------------------------------------------------------------------
# NM vs p  — sweep using MC  (fewer samples per point is fine for the trend)
# -----------------------------------------------------------------------
p_range       = np.logspace(-4, -1, 20)
MC_SWEEP      = 200_000   # samples per p-value during sweep
nm_full_arr   = []
nm_trunc_arr  = []

print(f'\nSweeping p  (MC, {MC_SWEEP} samples/point, identity gate) ...')
for pv in p_range:
    d_f, _ = monte_carlo_distribution(code, gate, pv, 'full',
                                       n_samples=MC_SWEEP, seed=0)
    d_t, _ = monte_carlo_distribution(code, gate, pv, 'truncated',
                                       n_samples=MC_SWEEP, seed=1)
    nm_f, _ = non_markovianity(marginal_L_given_sin(d_f))
    nm_t, _ = non_markovianity(marginal_L_given_sin(d_t))
    nm_full_arr.append(nm_f)
    nm_trunc_arr.append(nm_t)
    print(f'  p={pv:.4f}  NM(full)={nm_f:.3e}  NM(trunc)={nm_t:.3e}')

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(p_range, nm_full_arr,  'o-',  label='Full exRec',      color='steelblue')
ax.loglog(p_range, nm_trunc_arr, 's--', label='Truncated exRec', color='tomato')
ax.set_xlabel('Physical error rate  p', fontsize=11)
ax.set_ylabel('Non-Markovianity  NM (avg TVD)', fontsize=11)
ax.set_title(f'{code.name} — {gate.__name__}\nSyndrome-dependence of logical error vs p', fontsize=11)
ax.legend(); ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()
